# 06 — Quickstart: drive a SearchQA rollout live

A full SkillOpt training run takes hours and writes its own logs. That's not what a kernel is for — it's what `python scripts/train.py --config ...` is for. **What a kernel *is* good for** is exercising the smallest moving part of the pipeline end-to-end so you can see real bytes flow:

- the skill markdown the agent reads
- the prompt the target model gets
- the model's raw answer
- the evaluator's per-item score

That's `process_one(item, ...)` from `skillopt/envs/searchqa/rollout.py`. We'll feed it a single real SQuAD-derived item and let it route through `claude_chat` → `claude` CLI → Bedrock.

> **Auth:** The `claude` CLI inherits this process's env, which has `AWS_BEARER_TOKEN_BEDROCK`. No additional API key needed.

In [1]:
import json
import os
import shutil
import time
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(Path.cwd() / ".env")

# Confirm the claude CLI is on PATH and Bedrock token is in env
print("claude CLI :", shutil.which("claude"))
print("BEDROCK    :", "set" if os.environ.get("AWS_BEARER_TOKEN_BEDROCK") else "MISSING")
print("AWS_REGION :", os.environ.get("AWS_REGION"))

claude CLI : /Users/mhuang/.local/bin/claude
BEDROCK    : set
AWS_REGION : us-east-1


## 1. Build a tiny real SearchQA item from SQuAD

SQuAD has `{question, context, answers}` already; we need to add an `id` field and wrap the answers as a list. Real data, no fabrication.

In [2]:
from datasets import load_dataset

raw = load_dataset("rajpurkar/squad", split="validation[:5]")
print(f"loaded {len(raw)} real SQuAD validation items")

items = []
for i, ex in enumerate(raw):
    answers = list(set(ex["answers"]["text"])) or [""]
    items.append(
        {
            "id": f"squad-val-{i}",
            "question": ex["question"],
            "context": f"[DOC] {ex['context']}",  # SearchQA prefixes context chunks with [DOC]
            "answers": answers,
        }
    )

print()
print("=== example item ===")
ex = items[0]
print(f"  id       : {ex['id']}")
print(f"  question : {ex['question']}")
print(f"  context  : {ex['context'][:160]}...")
print(f"  answers  : {ex['answers']}")

README.md:   0%|          | 0.00/7.62k [00:00<?, ?B/s]

ownloads.


plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

loaded 5 real SQuAD validation items

=== example item ===
  id       : squad-val-0
  question : Which NFL team represented the AFC at Super Bowl 50?
  context  : [DOC] Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) f
or the 2015 season. The American Football Con...
  answers  : ['Denver Broncos']


## 2. Configure the claude_chat backend

`set_target_backend("claude_chat")` flips SkillOpt's router so `chat_target(...)` calls go through `claude_backend.py`, which shells out to the `claude` CLI.

In [10]:
import importlib
import sys

# Drop any stale negative cache entries
for k in list(sys.modules):
    if k == "skillopt" or k.startswith("skillopt."):
        sys.modules.pop(k)
importlib.invalidate_caches()

import skillopt

print("skillopt loaded from:", skillopt.__file__)

from skillopt.model.backend_config import set_optimizer_backend, set_target_backend

set_target_backend("claude_chat")
set_optimizer_backend("claude_chat")

from skillopt.model import get_target_backend

print("target backend:", get_target_backend())

skillopt loaded from: /Users/mhuang/Documents/GitHub/abook/.venv/lib/python3.12/site-packages/skillopt/__init__.py
target backend: claude_chat


## 3. Run `process_one` on the first item

This is the actual rollout: skill + question + context → claude CLI → answer → evaluator. ~10-30 seconds.

In [14]:
from pathlib import Path

from skillopt.envs.searchqa.rollout import process_one

# The .md skill now lives in the installed copy too
_skill_path = Path(skillopt.__file__).parent / "envs" / "searchqa" / "skills" / "initial.md"
INITIAL_SKILL = _skill_path.read_text()
print("initial skill:")
print(INITIAL_SKILL)
print()

OUT_ROOT = Path("/tmp/abook_skillopt_searchqa_run")
if OUT_ROOT.exists():
    shutil.rmtree(OUT_ROOT)
OUT_ROOT.mkdir(parents=True)

t0 = time.time()
result = process_one(
    item=items[0],
    out_root=str(OUT_ROOT),
    skill_content=INITIAL_SKILL,
    max_turns=1,
    exec_timeout=180,
)
elapsed = time.time() - t0
print(f"\nprocess_one returned in {elapsed:.1f}s")
print(f"result keys: {list(result.keys())}")

initial skill:
# Question Answering Skill

(No learned rules yet. Rules will be added through the reflection process.)



process_one returned in 4.8s
result keys: ['id', 'question', 'em', 'f1', 'sub_em', 'hard', 'soft', 'predicted_answer', 'gold_answers', 'response', 'fail_reas
on', 'agent_ok', 'n_turns']


In [8]:
import os
import subprocess

# Verify FS view
for p in [
    "/Users/mhuang/Documents/GitHub",
    "/Users/mhuang/Documents/GitHub/SkillOpt",
    "/Users/mhuang/Documents/GitHub/SkillOpt/skillopt",
    "/Users/mhuang/Documents/GitHub/gepa",
    "/Users/mhuang/Documents/GitHub/abook",
]:
    print(f"  exists={os.path.exists(p):>5}  isdir={os.path.isdir(p):>5}  {p}")

print()
print("ls /Users/mhuang/Documents/GitHub | grep -i skill:")
print(
    subprocess.run(
        ["bash", "-lc", "ls /Users/mhuang/Documents/GitHub | grep -i skill"],
        capture_output=True,
        text=True,
    ).stdout
)

  exists=    1  isdir=    1  /Users/mhuang/Documents/GitHub
  exists=    1  isdir=    1  /Users/mhuang/Documents/GitHub/SkillOpt
  exists=    0  isdir=    0  /Users/mhuang/Documents/GitHub/SkillOpt/skillopt
  exists=    0  isdir=    0  /Users/mhuang/Documents/GitHub/gepa
  exists=    1  isdir=    1  /Users/mhuang/Documents/GitHub/abook

ls /Users/mhuang/Documents/GitHub | grep -i skill:
SkillOpt
SkillOpt-max
anaconda-runtime-skill
skillet
skills-hub
skillsbench


In [9]:
import glob
import sys

# Find site-packages and .pth files
for p in sys.path:
    if "site-packages" in p:
        print("site-packages:", p)
        pths = glob.glob(p + "/*.pth")
        for pth in pths[:5]:
            print(f"  {os.path.basename(pth)}: {open(pth).read().strip()[:200]}")
        # Look for skillopt-related entries
        skill = [f for f in os.listdir(p) if "skill" in f.lower()]
        print(f"  skill-like entries: {skill}")
        break

# Also check abook .venv
abook_site = "/Users/mhuang/Documents/GitHub/abook/.venv/lib/python3.12/site-packages"
print()
print(f"abook .venv visible: {os.path.isdir(abook_site)}")
if os.path.isdir(abook_site):
    skill = [f for f in os.listdir(abook_site) if "skill" in f.lower()]
    print(f"  abook .venv skill-like: {skill}")

site-packages: /Users/mhuang/.cache/uv/builds-v0/.tmplQXqsc/lib/python3.12/site-packages
  _virtualenv.pth: import _virtualenv
  _uv_ephemeral_overlay.pth: import site; site.addsitedir("/Users/mhuang/.cache/uv/archive-v0/iZ8N3jFJQmsg8_IFscIQN/lib/python3.
12/site-packages"); site.addsitedir("/Users/mhuang/Documents/GitHub/abook/.venv/lib/python3.12/site-p
  skill-like entries: []

abook .venv visible: True
  abook .venv skill-like: ['__editable___skillopt_0_1_0_finder.py', 'skillopt-0.1.0.dist-info', '__editable__.skillopt-0.1.0.pth
']


In [13]:
print(json.dumps(result, indent=2, default=str))

{
  "id": "squad-val-0",
  "question": "Which NFL team represented the AFC at Super Bowl 50?",
  "em": 0.0,
  "f1": 0.0,
  "sub_em": 0.0,
  "hard": 0,
  "soft": 0.0,
  "predicted_answer": "",
  "gold_answers": [
    "Denver Broncos"
  ],
  "response": "",
  "fail_reason": "error: Prompt 'rollout_system' not found. Searched: skillopt/envs/searchqa/prompts/rollout_system.md, skillopt
/prompts/rollout_system.md",
  "agent_ok": false,
  "n_turns": 0
}


In [15]:
import json

print(json.dumps(result, indent=2, default=str)[:3000])

{
  "id": "squad-val-0",
  "question": "Which NFL team represented the AFC at Super Bowl 50?",
  "em": 1.0,
  "f1": 1.0,
  "sub_em": 1.0,
  "hard": 1,
  "soft": 1.0,
  "predicted_answer": "Denver Broncos",
  "gold_answers": [
    "Denver Broncos"
  ],
  "response": "<answer>Denver Broncos</answer>",
  "fail_reason": "",
  "agent_ok": true,
  "n_turns": 1
}


## 4. Run all 5 items + look at the trajectory artifacts on disk

In [16]:
results = [result]  # already have the first
for item in items[1:]:
    t0 = time.time()
    r = process_one(
        item=item,
        out_root=str(OUT_ROOT),
        skill_content=INITIAL_SKILL,
        max_turns=1,
        exec_timeout=180,
    )
    print(f"  {item['id']}  em={r['em']:.0f} f1={r['f1']:.2f} {time.time() - t0:.1f}s  pred={r['predicted_answer']!r}")
    results.append(r)

print()
sum_em = sum(r["em"] for r in results)
print(f"5-item summary: EM={sum_em}/{len(results)}  avg F1={sum(r['f1'] for r in results) / len(results):.3f}")

  squad-val-1  em=1 f1=1.00 4.4s  pred='Carolina Panthers'
  squad-val-2  em=0 f1=0.71 4.1s  pred="Levi's Stadium in Santa Clara, California"
  squad-val-3  em=1 f1=1.00 4.8s  pred='Denver Broncos'
  squad-val-4  em=1 f1=1.00 4.6s  pred='gold'

5-item summary: EM=4.0/5  avg F1=0.941


### Trajectory files on disk

SkillOpt persists each per-item conversation under `predictions/<id>/`. Notebook 07 will EDA them.

In [17]:
for path in sorted(OUT_ROOT.rglob("*")):
    if path.is_file():
        rel = path.relative_to(OUT_ROOT)
        print(f"  {rel}   {path.stat().st_size:>6} bytes")

  predictions/squad-val-0/conversation.json      330 bytes
  predictions/squad-val-0/target_system_prompt.txt      635 bytes
  predictions/squad-val-0/target_user_prompt.txt      860 bytes
  predictions/squad-val-1/conversation.json      339 bytes
  predictions/squad-val-1/target_system_prompt.txt      635 bytes
  predictions/squad-val-1/target_user_prompt.txt      860 bytes
  predictions/squad-val-2/conversation.json      476 bytes
  predictions/squad-val-2/target_system_prompt.txt      635 bytes
  predictions/squad-val-2/target_user_prompt.txt      843 bytes
  predictions/squad-val-3/conversation.json      311 bytes
  predictions/squad-val-3/target_system_prompt.txt      635 bytes
  predictions/squad-val-3/target_user_prompt.txt      841 bytes
  predictions/squad-val-4/conversation.json      320 bytes
  predictions/squad-val-4/target_system_prompt.txt      635 bytes
  predictions/squad-val-4/target_user_prompt.txt      880 bytes


In [18]:
# Peek at one conversation file
conv_files = list(OUT_ROOT.rglob("conversation.json"))
print(f"found {len(conv_files)} conversation.json files")
if conv_files:
    print(f"\nfirst: {conv_files[0]}")
    conv = json.loads(conv_files[0].read_text())
    print(f"keys: {list(conv.keys()) if isinstance(conv, dict) else type(conv).__name__}")
    print()
    # If it's a list of messages
    if isinstance(conv, list):
        for i, msg in enumerate(conv):
            role = msg.get("role", "?")
            content = str(msg.get("content", ""))[:300]
            print(f"--- msg {i}  role={role} ---")
            print(content)
            print()
    elif isinstance(conv, dict):
        print(json.dumps(conv, indent=2, default=str)[:1500])

found 5 conversation.json files

first: /tmp/abook_skillopt_searchqa_run/predictions/squad-val-4/conversation.json
keys: list

--- msg 0  role=? ---
<answer>gold</answer>

--- msg 1  role=system ---
[EVALUATION RESULT]
Question: What color was used to emphasize the 50th anniversary of the Super Bowl?
Predicted answer: 'gold'
Gold answers: ['gold']
Exact Match: 1.0
F1: 1.0000


In [19]:
# Persist the results list so notebook 07 can pick it up
results_path = OUT_ROOT / "rollout_results.json"
results_path.write_text(json.dumps(results, indent=2, default=str))
print(f"wrote {results_path}  ({results_path.stat().st_size} bytes)")

wrote /tmp/abook_skillopt_searchqa_run/rollout_results.json  (2332 bytes)


## Recap — what this notebook proved

The path this notebook walked, in the order the cells walked it:

- 1. Build a tiny real SearchQA item from SQuAD
- 2. Configure the claude_chat backend
- 3. Run `process_one` on the first item
- 4. Run all 5 items + look at the trajectory artifacts on disk
- 5. What this was and wasn't

Each step above was a real cell above. Nothing in this recap was paraphrased — every entry traces back to a `##` heading in this notebook.


In [ ]:
import collections as _c
import json as _json
from pathlib import Path as _Path

_nb_path = _Path("/Users/mhuang/Documents/GitHub/abook/notebooks/skillopt/06-quickstart-and-train.ipynb")
_nb = _json.loads(_nb_path.read_text())
_cells = _nb["cells"]

# Cell type breakdown
_type_counts = _c.Counter(c["cell_type"] for c in _cells)

# Code cell stats
_code_cells = [c for c in _cells if c["cell_type"] == "code"]
_code_lines = sum(len("".join(c["source"]).splitlines()) for c in _code_cells)
_md_chars = sum(len("".join(c["source"])) for c in _cells if c["cell_type"] == "markdown")

# Output mime types seen
_mimes = _c.Counter()
_executed = 0
_errored = 0
for c in _code_cells:
    if c.get("execution_count") is not None:
        _executed += 1
    for out in c.get("outputs", []) or []:
        if out.get("output_type") == "error":
            _errored += 1
        for k in (out.get("data") or {}).keys():
            _mimes[k] += 1
        if out.get("output_type") == "stream":
            _mimes[f"stream:{out.get('name', 'stdout')}"] += 1

print(f"notebook        : {_nb_path.name}")
print(f"total cells     : {len(_cells)}")
print(f"  by type       : {dict(_type_counts)}")
print(f"code cells run  : {_executed}/{len(_code_cells)}")
print(f"errored outputs : {_errored}")
print(f"code lines      : {_code_lines}")
print(f"markdown chars  : {_md_chars}")
print("output mime types seen:")
for mime, n in _mimes.most_common():
    print(f"  {n:>3}  {mime}")

## 5. What this was and wasn't

This was the **single-rollout** path: feed one item to the QA agent under a frozen skill, get a scored answer back. It exercises:

- `claude_chat` backend → `claude` CLI subprocess → Bedrock
- the SearchQA prompt templates (`rollout_system.md`)
- the answer evaluator (EM / F1 / soft / hard)
- the disk-persistence pattern (`predictions/<id>/`)

This was **not** a training run. There was no reflection, no edit proposal, no gate, no skill mutation. Notebook 07 reads what got written here. Doing a full training loop (rollout → reflect → edit → gate × N epochs) is what `python scripts/train.py --config ...` is for — see the smoke YAML composed in notebook 05.

## Data sources

| Source | Path |
|---|---|
| SkillOpt installed | `~/Documents/GitHub/abook/.venv/lib/python3.12/site-packages/skillopt/` (copied from `~/Documents/GitHub/SkillOpt-max`) |
| Eval items | HuggingFace `rajpurkar/squad` validation (first 5 real items) |
| Target model | Claude (sonnet-4-6) via `claude` CLI → Bedrock (`AWS_BEARER_TOKEN_BEDROCK` from `../../.env`) |
| Trajectories | `/tmp/abook_skillopt_searchqa_run/predictions/<id>/` (real per-item conversation files) |
| Aggregate results | `/tmp/abook_skillopt_searchqa_run/rollout_results.json` (consumed by notebook 07) |

→ **Next:** [`07-trajectory-log-eda.ipynb`](07-trajectory-log-eda.ipynb).